# 51. 예측 결과 시각화와 문제 찾기

50장에서 학습한 baseline 모델의 예측 결과를 직접 봅니다. 숫자 metric만 보지 않고 image, target, prediction, overlay, error map을 함께 저장합니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "seg7_utils.py").exists():
    NOTEBOOK_DIR = Path("Vision 기초/7장")

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "mini_shapes_seg"
RUNS_ROOT = NOTEBOOK_DIR / "runs"

from seg7_utils import *
set_korean_font()
set_seed(7)

## 51-1. Baseline checkpoint 불러오기

In [ ]:
torch, nn, F, DataLoader, Dataset = require_torch()
MiniShapesDataset = build_dataset_class()
models = build_models()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
val_ds = MiniShapesDataset(DATA_ROOT, "val", augment=False)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

model = models["tiny_fcn"](num_classes=len(CLASS_NAMES), channels=8).to(device)
checkpoint_path = RUNS_ROOT / "50_baseline_fcn" / "checkpoint.pt"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
criterion = nn.CrossEntropyLoss()
metrics = evaluate(model, val_loader, criterion, device, len(CLASS_NAMES))
metrics

## 51-2. 예측 결과와 error map 저장

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

run_dir = RUNS_ROOT / "51_visual_error_check"
pred_dir = run_dir / "pred_samples"
pred_dir.mkdir(parents=True, exist_ok=True)

model.eval()
error_ratios = []
with torch.no_grad():
    for idx in range(min(6, len(val_ds))):
        image_tensor, target_tensor = val_ds[idx]
        logits = model(image_tensor.unsqueeze(0).to(device))
        pred = logits.argmax(dim=1).squeeze(0).cpu().numpy()
        target = target_tensor.numpy()
        image = (image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        error = pred != target
        error_ratios.append(float(error.mean()))

        save_prediction_grid(pred_dir / f"sample_{idx:02d}.png", image, target, pred, title="baseline error check")

        fig, axes = plt.subplots(1, 2, figsize=(6, 3))
        axes[0].imshow(make_overlay(image, pred))
        axes[0].set_title("prediction overlay")
        axes[1].imshow(error, cmap="gray")
        axes[1].set_title("error map")
        for ax in axes:
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(pred_dir / f"error_{idx:02d}.png", dpi=140)
        plt.show()

metrics["experiment_name"] = "51_visual_error_check"
metrics["mean_error_ratio_samples"] = float(np.mean(error_ratios))
save_json(run_dir / "config.json", {
    "experiment_name": "51_visual_error_check",
    "source_checkpoint": str(checkpoint_path),
    "model_name": "tiny_fcn",
    "loss_name": "ce",
    "note": "baseline prediction visualization and error maps",
})
save_json(run_dir / "metrics.json", metrics)
metrics